# Verifier-Guided Code Generation — Main Experiment Notebook

This notebook is the single run log for the project. It is structured to match the paper and to keep **full-set (system-level)** results separate from **held-out (generalization)** evidence.

**Two evaluation views**
- **Full-set (system-level) view:** evaluate selection policies on all tasks present in the candidate/test files. This is useful to understand the end-to-end pipeline, but it can include tasks that were used during pairwise data construction.
- **Held-out (generalization) view:** evaluate the ranker only on *selection-relevant* tasks held out at the task level (tasks that have both passing and failing candidates so pairwise ranking is meaningful). These results are the main evidence that the learned selector generalizes.

All artifacts are written under `outputs/`:
- `outputs/candidates/`: sampled programs (JSONL)
- `outputs/tests/`: unit-test outcomes (JSONL)
- `outputs/ranker_data/`: pairwise datasets + checkpoints
- `outputs/results/`: baseline CSVs + held-out logs


In [ ]:
!pip -q install "transformers>=4.41.0" "accelerate>=0.30.0" "datasets>=2.19.0" "pyyaml>=6.0" "tqdm>=4.66.0" "torch>=2.1.0"


## 0) Run manifest

This keeps the notebook readable and avoids hard-coded filenames scattered across cells.


In [ ]:
# === Update these after each run (copy paths from the script outputs) it goes step by step so you can see the same progress with the paper, I decided to turning back here to update the paths easier than the confusion middle of the notebook ===

# Smoke test (optional)
CANDS_SMOKE = "outputs/candidates/humaneval_candidates_YYYYMMDD_HHMMSS.jsonl"
TESTS_SMOKE = "outputs/tests/humaneval_tests_YYYYMMDD_HHMMSS.jsonl"

# Full-set N=8 run
CANDS_N8 = "outputs/candidates/humaneval_candidates_YYYYMMDD_HHMMSS.jsonl"
TESTS_N8 = "outputs/tests/humaneval_tests_YYYYMMDD_HHMMSS.jsonl"

# Full-set N=16 run (RQ3)
CANDS_N16 = "outputs/candidates/humaneval_candidates_YYYYMMDD_HHMMSS.jsonl"
TESTS_N16 = "outputs/tests/humaneval_tests_YYYYMMDD_HHMMSS.jsonl"

# Prompt-conditioned pair datasets (the ones used to train the main ranker)
PAIRS_PROMPT_TRAIN = "outputs/ranker_data/pairs_prompt_train_YYYYMMDD_HHMMSS.jsonl"
PAIRS_PROMPT_VAL   = "outputs/ranker_data/pairs_prompt_val_YYYYMMDD_HHMMSS.jsonl"
PAIRS_PROMPT_TEST  = "outputs/ranker_data/pairs_prompt_test_YYYYMMDD_HHMMSS.jsonl"

# Trained ranker checkpoint used in the paper
RANKER_CKPT = "outputs/ranker_data/ranker_model_YYYYMMDD_HHMMSS/best.pt"


In [ ]:
# Make the path variables available to shell cells as environment variables
import os
for k in [
    "CANDS_SMOKE","TESTS_SMOKE",
    "CANDS_N8","TESTS_N8",
    "CANDS_N16","TESTS_N16",
    "PAIRS_PROMPT_TRAIN","PAIRS_PROMPT_VAL","PAIRS_PROMPT_TEST",
    "RANKER_CKPT",
]:
    os.environ[k] = globals().get(k, "")


## 1) Smoke test (optional)

I used this when I changed or updated the code. It checks end-to-end wiring on a tiny subset and its fast.


In [ ]:
!python -m src.generate_candidates --config configs/exp.yaml


In [ ]:
!python -m src.run_tests \
  --config configs/exp.yaml \
  --candidates {CANDS_SMOKE} \
  --timeout_s 3.0


In [ ]:
!python -m src.evaluate_baselines \
  --config configs/exp.yaml \
  --candidates outputs/candidates/humaneval_candidates_20260304_133744.jsonl \
  --tests outputs/tests/humaneval_tests_20260304_134549.jsonl


## 2) Full-set run (HumanEval, N=8)

Before running generation, set `sampling.n_candidates: 8` in `configs/exp.yaml`.

This produces the main artifacts for RQ1/RQ2 (system view) and the training data for the ranker.


In [ ]:
!python -m src.generate_candidates --config configs/exp.yaml


In [ ]:
!python -m src.run_tests \
  --config configs/exp.yaml \
  --candidates {CANDS_N8} \
  --timeout_s 3.0


In [ ]:
!wc -l {TESTS_N8}


In [ ]:
!python -m src.evaluate_baselines \
  --config configs/exp.yaml \
  --candidates {CANDS_N8} \
  --tests {TESTS_N8}


## 3) Build prompt-conditioned pairs and train the ranker (main model)

This is the **only learned component**. The generator stays fixed.

Pairs are built only for tasks where ranking is meaningful (tasks with both passing and failing candidates).


In [ ]:
!python -m src.build_ranker_dataset \
  --config configs/exp.yaml \
  --candidates {CANDS_N8} \
  --tests {TESTS_N8} \
  --pairs_per_task 120 \
  --seed 42


In [ ]:
!ls outputs/ranker_data | grep pairs_prompt | tail -n 20


In [ ]:
!python -m src.train_ranker \
  --config configs/exp.yaml \
  --train_path outputs/ranker_data/pairs_prompt_train_20260317_153223.jsonl \
  --val_path outputs/ranker_data/pairs_prompt_val_20260317_153223.jsonl \
  --test_path outputs/ranker_data/pairs_prompt_test_20260317_153223.jsonl \
  --encoder_name microsoft/codebert-base \
  --max_length 512 \
  --batch_size 8 \
  --epochs 6 \
  --lr 2e-5 \
  --fp16


## 4) Full-set evaluation (system view) on N=8 candidates

This evaluates policies on **all tasks present** in the candidate/test files.  
Used this for the descriptive end-to-end view (RQ1/RQ2 tables on 164 tasks).

For generalization evidence, see Section 6 (held-out evaluation).


In [ ]:
!python -m src.evaluate_ranker_selection \
  --config configs/exp.yaml \
  --candidates {CANDS_N8} \
  --tests {TESTS_N8} \
  --ckpt {RANKER_CKPT} \
  --encoder_name microsoft/codebert-base \
  --max_length 512 \
  --k_list 1,2,4,8


### Error analysis (N=8)

This separates generation-limited tasks (no passing candidate) from ranker misses.


In [ ]:
!python -m src.error_analysis_shortlist \
  --candidates {CANDS_N8} \
  --tests {TESTS_N8} \
  --ckpt {RANKER_CKPT}


## 5) Generalization check (system view) with N=16 candidates (RQ3)

Before running generation, set `sampling.n_candidates: 16` in `configs/exp.yaml`.

This uses a **fresh** candidate pool but the **same** trained ranker checkpoint.


In [ ]:
!python -m src.generate_candidates --config configs/exp.yaml


In [ ]:
!python -m src.run_tests \
  --config configs/exp.yaml \
  --candidates {CANDS_N16} \
  --timeout_s 3.0


In [ ]:
!wc -l {TESTS_N16}


In [ ]:
!python -m src.evaluate_baselines \
  --config configs/exp.yaml \
  --candidates {CANDS_N16} \
  --tests {TESTS_N16}


In [ ]:
!python -m src.evaluate_ranker_selection \
  --config configs/exp.yaml \
  --candidates {CANDS_N16} \
  --tests {TESTS_N16} \
  --ckpt {RANKER_CKPT} \
  --encoder_name microsoft/codebert-base \
  --max_length 512 \
  --k_list 1,2,4,8,16


In [ ]:
!python -m src.error_analysis_shortlist \
  --candidates {CANDS_N16} \
  --tests {TESTS_N16} \
  --ckpt {RANKER_CKPT}


## 6) Held-out generalization evaluation on selection-relevant tasks (main evidence)

Why a selection-relevant held-out subset:
- Pairwise ranking supervision exists only for tasks with both passing and failing candidates.
- Tasks with only passing or only failing candidates cannot produce (pass, fail) pairs, so they are not ranking examples under this setup.
- Therefore, the held-out evaluation focuses on the subset where the selector problem actually exists.

Below, each run:
1) builds prompt-conditioned pairs with a random task split,
2) trains a ranker on the train split,
3) evaluates only on held-out selection-relevant tasks from the split’s test pairs file.


In [ ]:
# Build a fresh prompt-conditioned dataset from the N=16 artifacts (I recommend this for held-out evaluation otherwise the results will be biased and might be different from the paper)
!python -m src.build_ranker_dataset --config configs/exp.yaml \
  --candidates {CANDS_N16} \
  --tests {TESTS_N16} \
  --pairs_per_task 120 --seed 42


In [ ]:
# Sanity check: task-level splits are disjoint for the current (latest) prompt-pairs files.
import json, glob

def uniq(path):
    s=set()
    with open(path,'r') as f:
        for line in f:
            r=json.loads(line)
            s.add(r["task_id"])
    return s

train = uniq(PAIRS_PROMPT_TRAIN)
val   = uniq(PAIRS_PROMPT_VAL)
test  = uniq(PAIRS_PROMPT_TEST)

print("unique tasks in train pairs:", len(train))
print("unique tasks in val pairs:", len(val))
print("unique tasks in test pairs:", len(test))
print("train∩val:", len(train & val), "train∩test:", len(train & test), "val∩test:", len(val & test))


In [ ]:
%%bash
# Uses the N=16 candidate/test artifacts and runs 5 random task splits.
# Python manifest cell must be filled with the paths.


for SEED in 0 1 2 3 4; do
  echo "===== SEED $SEED ====="

  python -m src.build_ranker_dataset --config configs/exp.yaml \
    --candidates $CANDS_N16 \
    --tests $TESTS_N16 \
    --pairs_per_task 120 --seed ${SEED} \
    --train_frac 0.55 --val_frac 0.10

  TRAIN=$(ls -t outputs/ranker_data/pairs_prompt_train_*.jsonl | head -1)
  VAL=$(ls -t outputs/ranker_data/pairs_prompt_val_*.jsonl | head -1)
  TEST=$(ls -t outputs/ranker_data/pairs_prompt_test_*.jsonl | head -1)

  python -m src.train_ranker \
    --config configs/exp.yaml \
    --train_path ${TRAIN} \
    --val_path ${VAL} \
    --test_path ${TEST} \
    --encoder_name microsoft/codebert-base \
    --batch_size 16 --epochs 6 --max_length 384 --lr 2e-5 --fp16

  CKPTDIR=$(ls -td outputs/ranker_data/ranker_model_* | head -1)

  python -m src.evaluate_ranker_selection \
    --config configs/exp.yaml \
    --candidates $CANDS_N16 \
    --tests $TESTS_N16 \
    --ckpt ${CKPTDIR}/best.pt \
    --encoder_name microsoft/codebert-base \
    --max_length 512 \
    --k_list 1,2,4,8,16 \
    --task_ids_from_pairs ${TEST} \
    | tee outputs/results/heldout_seed${SEED}.txt
done


### Aggregate held-out results (mean ± std over runs)

This reads the saved `heldout_seed*.txt` logs produced above and reports mean and std.


In [ ]:
import glob, re, statistics as st

files = sorted(glob.glob("outputs/results/heldout_seed*.txt"))
if not files:
    raise SystemExit("No heldout_seed*.txt files found.")

def extract(path):
    txt=open(path,'r').read()
    m_tasks=re.search(r"Tasks evaluated:\s*(\d+)", txt)
    tasks=int(m_tasks.group(1)) if m_tasks else None

    def grab(key):
        m=re.search(rf"{re.escape(key)}:\s*([0-9.]+)", txt)
        return float(m.group(1)) if m else None

    out = {
        "tasks": tasks,
        "first": grab("pass@1 first-sample"),
        "best": grab("pass@1 test-all best-of-N"),
        "top1": grab("pass@1 ranker top-1"),
        "top2": grab("pass@1 ranker top-2+tests"),
        "top4": grab("pass@1 ranker top-4+tests"),
    }
    return out

rows=[extract(f) for f in files]

def ms(key):
    vals=[r[key] for r in rows if r[key] is not None]
    return st.mean(vals), (st.pstdev(vals) if len(vals)>1 else 0.0)

print("Runs:", len(rows))
print("Avg held-out tasks:", st.mean([r["tasks"] for r in rows]))

for k in ["first","best","top1","top2","top4"]:
    m,s = ms(k)
    print(f"{k}: {m:.3f} ± {s:.3f}")
